In [6]:
import pandas as pd
from pathlib import Path

Pathway 

In [ ]:
press_conf_fed = Path('/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/Fed Side/Excel Sentences Results')
press_conf = pd.read_excel( press_conf_fed / "fed_statements_presconf_WITH_LLM_LABELS_2021.xlsx")
pre

Analysis

In [ ]:
press_conf.columns

Index(['source', 'orig_row', 'id', 'raw_text', 'doc_type', 'clean_text',
       'sentence', 'Dissagreement', 'label', 'Unnamed: 9', 'llm_majority',
       'claude-haiku__p1_analyst_direct', 'claude-haiku__p2_analyst_rubric',
       'claude-haiku__p3_analyst_rules', 'claude-haiku__p4_analyst_json',
       'claude-haiku__p5_analyst_fewshot', 'claude-sonnet__p1_analyst_direct',
       'claude-sonnet__p2_analyst_rubric', 'claude-sonnet__p3_analyst_rules',
       'claude-sonnet__p4_analyst_json', 'claude-sonnet__p5_analyst_fewshot',
       'deepseek-chat__p1_analyst_direct', 'deepseek-chat__p2_analyst_rubric',
       'deepseek-chat__p3_analyst_rules', 'deepseek-chat__p4_analyst_json',
       'deepseek-chat__p5_analyst_fewshot',
       'deepseek-reasoner__p1_analyst_direct',
       'deepseek-reasoner__p2_analyst_rubric',
       'deepseek-reasoner__p3_analyst_rules',
       'deepseek-reasoner__p4_analyst_json',
       'deepseek-reasoner__p5_analyst_fewshot',
       'gpt-5-mini__p1_analyst_dir

In [10]:
"""
Analyze where each LLM/prompt combination disagrees with the true (human) label,
then summarize — per model (pooled across its 5 prompts) — which category
(the 'Dissagreement' column, e.g. Economic Status, Labor, Fed Expectations, etc.)
it misclassifies most often.

Expects a dataframe `press_conf` with columns like:
    'label'                              -> ground-truth class (D/N/H)
    'Dissagreement'                      -> category tag for the sentence
    '<model>__<prompt>'                  -> model's predicted class for that prompt
"""

import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------
press_conf_fed = Path(
    '/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/'
    'BdF-Project/Extension LLM Classification Task/Fed Side/Excel Sentences Results'
)
press_conf = pd.read_excel(
    press_conf_fed / "fed_statements_presconf_WITH_LLM_LABELS_2021.xlsx"
)

TRUE_LABEL_COL = "label"
CATEGORY_COL = "Dissagreement"

# ---------------------------------------------------------------
# 2. Identify all model/prompt prediction columns automatically
#    (anything with the "<model>__<prompt>" naming pattern)
# ---------------------------------------------------------------
pred_cols = [c for c in press_conf.columns if "__" in c]

# Keep only rows that actually have a ground-truth label
df = press_conf.dropna(subset=[TRUE_LABEL_COL]).copy()

# ---------------------------------------------------------------
# 3. Build a long/tidy "miss table": one row per (sentence, model, prompt)
#    where the prediction != true label
# ---------------------------------------------------------------
records = []
for col in pred_cols:
    model, prompt = col.split("__", 1)
    sub = df[[TRUE_LABEL_COL, CATEGORY_COL, col]].copy()
    sub = sub.rename(columns={col: "pred"})
    sub["model"] = model
    sub["prompt"] = prompt
    sub["sentence_id"] = df.index  # or df['id'] if you prefer a stable id
    records.append(sub)

long_df = pd.concat(records, ignore_index=True)

# Drop rows where the model didn't produce a prediction at all (NaN)
long_df = long_df.dropna(subset=["pred"])

# Flag misses: prediction != true label
long_df["is_miss"] = long_df["pred"].str.strip().str.upper() != long_df[
    TRUE_LABEL_COL
].str.strip().str.upper()

misses = long_df[long_df["is_miss"]].copy()

# ---------------------------------------------------------------
# 4. Per model x prompt: how many misses, and by category
# ---------------------------------------------------------------
miss_counts_by_model_prompt = (
    misses.groupby(["model", "prompt", CATEGORY_COL])
    .size()
    .rename("n_misses")
    .reset_index()
    .sort_values(["model", "prompt", "n_misses"], ascending=[True, True, False])
)

print("=== Misses by model, prompt, and category ===")
print(miss_counts_by_model_prompt.to_string(index=False))

# ---------------------------------------------------------------
# 5. Per model, POOLED across all 5 prompts:
#    which category does each model misclassify the most?
# ---------------------------------------------------------------
miss_counts_by_model = (
    misses.groupby(["model", CATEGORY_COL])
    .size()
    .rename("n_misses")
    .reset_index()
)

# Top miscategorized category per model
top_category_per_model = (
    miss_counts_by_model.sort_values(["model", "n_misses"], ascending=[True, False])
    .groupby("model")
    .head(1)
    .reset_index(drop=True)
)

print("\n=== Top miscategorized category per model (pooled across prompts) ===")
print(top_category_per_model.to_string(index=False))

# Full ranked breakdown (all categories, ranked, per model) — often more useful
ranked_by_model = (
    miss_counts_by_model.sort_values(["model", "n_misses"], ascending=[True, False])
    .reset_index(drop=True)
)
print("\n=== Full ranked category miss-breakdown per model ===")
print(ranked_by_model.to_string(index=False))

# ---------------------------------------------------------------
# 5b. Same as above but normalized: error RATE within each category
#     (misses in that category / total predictions made in that category)
#     This matters because some categories (e.g. Economic Status) simply
#     have way more sentences than others (e.g. Key Words), so raw miss
#     counts alone are misleading.
# ---------------------------------------------------------------
# Total predictions made per model, per category (denominator)
total_by_model_cat = (
    long_df.groupby(["model", CATEGORY_COL])
    .size()
    .rename("n_total")
    .reset_index()
)

# Merge misses with totals to compute rate
rate_by_model_cat = miss_counts_by_model.merge(
    total_by_model_cat, on=["model", CATEGORY_COL], how="right"
)
rate_by_model_cat["n_misses"] = rate_by_model_cat["n_misses"].fillna(0)
rate_by_model_cat["error_rate"] = (
    rate_by_model_cat["n_misses"] / rate_by_model_cat["n_total"]
)

rate_by_model_cat = rate_by_model_cat.sort_values(
    ["model", "error_rate"], ascending=[True, False]
).reset_index(drop=True)

print("\n=== Category error RATE per model (misses / total in category) ===")
print(rate_by_model_cat.to_string(index=False))

# Top category by error RATE per model (i.e. the category the model is
# proportionally worst at, not just the one with the most raw misses)
top_category_by_rate_per_model = (
    rate_by_model_cat.sort_values(["model", "error_rate"], ascending=[True, False])
    .groupby("model")
    .head(1)
    .reset_index(drop=True)
)

print("\n=== Top category by ERROR RATE per model (pooled across prompts) ===")
print(top_category_by_rate_per_model.to_string(index=False))

# Optional: same breakdown but also split by prompt (rate within
# model x prompt x category)
total_by_model_prompt_cat = (
    long_df.groupby(["model", "prompt", CATEGORY_COL])
    .size()
    .rename("n_total")
    .reset_index()
)
rate_by_model_prompt_cat = miss_counts_by_model_prompt.merge(
    total_by_model_prompt_cat, on=["model", "prompt", CATEGORY_COL], how="right"
)
rate_by_model_prompt_cat["n_misses"] = rate_by_model_prompt_cat["n_misses"].fillna(0)
rate_by_model_prompt_cat["error_rate"] = (
    rate_by_model_prompt_cat["n_misses"] / rate_by_model_prompt_cat["n_total"]
)
rate_by_model_prompt_cat = rate_by_model_prompt_cat.sort_values(
    ["model", "prompt", "error_rate"], ascending=[True, True, False]
).reset_index(drop=True)

# ---------------------------------------------------------------
# 6. Optional: overall accuracy per model (pooled across prompts) for context
# ---------------------------------------------------------------
total_by_model = long_df.groupby("model").size().rename("n_total")
miss_by_model = misses.groupby("model").size().rename("n_miss")
acc_summary = pd.concat([total_by_model, miss_by_model], axis=1).fillna(0)
acc_summary["miss_rate"] = acc_summary["n_miss"] / acc_summary["n_total"]
acc_summary = acc_summary.sort_values("miss_rate")
print("\n=== Overall miss rate per model (pooled across prompts) ===")
print(acc_summary.to_string())

# ---------------------------------------------------------------
# 7. Save results to Excel for easy inspection
# ---------------------------------------------------------------
out_path = press_conf_fed / "model_disagreement_analysis.xlsx"
with pd.ExcelWriter(out_path) as writer:
    misses.to_excel(writer, sheet_name="all_misses", index=False)
    miss_counts_by_model_prompt.to_excel(
        writer, sheet_name="misses_by_model_prompt_cat", index=False
    )
    ranked_by_model.to_excel(writer, sheet_name="misses_by_model_cat", index=False)
    top_category_per_model.to_excel(
        writer, sheet_name="top_category_per_model", index=False
    )
    rate_by_model_cat.to_excel(writer, sheet_name="error_rate_by_model_cat", index=False)
    top_category_by_rate_per_model.to_excel(
        writer, sheet_name="top_category_by_rate", index=False
    )
    rate_by_model_prompt_cat.to_excel(
        writer, sheet_name="error_rate_by_model_prompt_cat", index=False
    )
    acc_summary.to_excel(writer, sheet_name="overall_miss_rate")

print(f"\nSaved detailed results to: {out_path}")

=== Misses by model, prompt, and category ===
            model             prompt       Dissagreement  n_misses
     claude-haiku  p1_analyst_direct     Economic Status       156
     claude-haiku  p1_analyst_direct               Labor        17
     claude-haiku  p1_analyst_direct     Foreign Nations         9
     claude-haiku  p1_analyst_direct Energy/House Prices         7
     claude-haiku  p1_analyst_direct   Fed Expectations          6
     claude-haiku  p1_analyst_direct             General         4
     claude-haiku  p1_analyst_direct        Money Supply         2
     claude-haiku  p1_analyst_direct Dollar Value Change         1
     claude-haiku  p2_analyst_rubric     Economic Status       162
     claude-haiku  p2_analyst_rubric               Labor        16
     claude-haiku  p2_analyst_rubric     Foreign Nations         7
     claude-haiku  p2_analyst_rubric Energy/House Prices         5
     claude-haiku  p2_analyst_rubric   Fed Expectations          5
     claude-haik

ECB Side